# Tau-måling: Qwen2.5-32B (RunPod A100)

**Krev A100 (40GB) — IKKJE T4**

Måler τ = r_eff / r_max frå skjulte tilstandar via SVD.

Goldilocks: [exp(−γ), 1/ζ(3)] = [0.5615, 0.8319]

Intra-familie skalering: Qwen2.5 7B→14B→32B

In [ ]:
# Installer avhengigheter
!pip install -q transformers accelerate bitsandbytes torch numpy

In [ ]:
# HuggingFace innlogging via Colab Secrets
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN lasta frå Colab Secrets.")
except Exception:
    print("Ingen HF_TOKEN — berre opne modellar.")

In [ ]:
import torch
import numpy as np
import math
from transformers import AutoTokenizer, AutoModel

GAMMA   = 0.5772156649
ZETA3   = 1.2020569032
TAU_MIN = math.exp(-GAMMA)
TAU_MAX = 1.0 / ZETA3

TEXTS = {
    "coherent":   "The structure of a system reveals itself through the patterns it sustains over time. Coherence emerges when local rules propagate consistently across all scales.",
    "random":     "Quantum entropy bicycle seventeen. Mountain glass decides purple. The concept flows between adjacent memory structures. Floating decisions cascade.",
    "repetitive": "the the the the the the the the the the the the the the the the the the the the the the the the",
}

def measure_tau(hidden):
    H = hidden.squeeze(0).float().cpu().numpy()
    _, s, _ = np.linalg.svd(H, full_matrices=False)
    s2 = s ** 2
    p  = s2 / s2.sum()
    p  = p[p > 1e-10]
    H_sp  = -np.sum(p * np.log(p))
    r_eff = np.exp(H_sp)
    r_max = min(H.shape)
    return r_eff / r_max

print(f"Goldilocks: [{TAU_MIN:.4f}, {TAU_MAX:.4f}}]")

In [ ]:
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
import os

MODEL_NAME = "Qwen/Qwen2.5-32B"
PREDICTION = 0.26  # intra-familie skaleringslov: 0.084 × 32^0.33 ≈ 0.265 (oppdatert etter 14B-måling)

bnb_config = BitsAndBytesConfig(load_in_4bit=True)
token = os.environ.get("HF_TOKEN")

print(f"Laster {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=token)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,
    token=token,
)
model.eval()
print("Klar.")

In [ ]:
# Mål tau
resultater = {}
for label, text in TEXTS.items():
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    tau = measure_tau(outputs.hidden_states[-1])
    resultater[label] = tau
    status = ("GOLDILOCKS ★" if TAU_MIN <= tau <= TAU_MAX
              else "BELOW" if tau < TAU_MIN else "ABOVE")
    print(f"[{label:>10}]  tau = {tau:.4f}  {status}")

print(f"\nPrediksjon:  {PREDICTION:.4f}")
print(f"Målt:        {resultater['coherent']:.4f}")
print(f"Avvik:       {abs(resultater['coherent'] - PREDICTION):.4f}")
print(f"\nRekkefølge (koherent > tilfeldig > repetitivt): "
      f"{'JA ✓' if resultater['coherent'] > resultater['random'] > resultater['repetitive'] else 'NEI ✗'}")